In [1]:
!pip uninstall -y transformers

# 2. Install the specific Gemma-3 supported branch
!pip install git+https://github.com/huggingface/transformers@v4.49.0-Gemma-3
!pip install git+https://github.com/huggingface/transformers
!pip install --no-cache-dir --upgrade --index-url https://download.pytorch.org/whl/cu118 torch torchvision torchaudio
!pip install -q -U datasets

Found existing installation: transformers 5.2.0
Uninstalling transformers-5.2.0:
  Successfully uninstalled transformers-5.2.0
  Cloning https://github.com/huggingface/transformers (to revision v4.49.0-Gemma-3) to /tmp/pip-req-build-io2gfnyj
  Running command git clone --filter=blob:none --quiet https://github.com/huggingface/transformers /tmp/pip-req-build-io2gfnyj
  Running command git checkout -q 1c0f782fe5f983727ff245c4c1b3906f9b99eec2
  Resolved https://github.com/huggingface/transformers to commit 1c0f782fe5f983727ff245c4c1b3906f9b99eec2
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 10.6 MB/s eta 0:00:00 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 68.1 MB/s eta 0:00:00:00:01
  Created wheel for transformers: filename=transformers-4.50.0.dev0-py3-none-any.whl size=10936460 sha256=5bc62fb737c2359b3699b4bc1fc10cd7324fc

In [2]:
from kaggle_secrets import UserSecretsClient
import huggingface_hub

try:
    user_secrets = UserSecretsClient()
    hf_token = user_secrets.get_secret("HF_TOKEN")
    huggingface_hub.login(token=hf_token)
    print("\n--- Authenticated with Hugging Face successfully! ---")
except Exception as e:
    print("\n--- Warning: Could not find HF_TOKEN in Kaggle Secrets ---")
    print("Please set it up via Add-ons -> Secrets to download Gemma.\n", e)


--- Authenticated with Hugging Face successfully! ---


In [3]:
import os
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODELS_DIR = "/kaggle/input/models/anurag2006/unlearning-models/transformers/long-tv-training/1/models"
BASE_MODEL_ID = "google/gemma-3-1b-it"

def get_available_models():
    """Scans the models directory for saved checkpoints."""
    available_models = ["Base Model (Download/Cache)"]
    if os.path.exists(MODELS_DIR):
        for item in os.listdir(MODELS_DIR):
            item_path = os.path.join(MODELS_DIR, item)
            if os.path.isdir(item_path) and "config.json" in os.listdir(item_path):
                available_models.append(item)
    return available_models

def load_inference_model(model_name):
    """Loads the selected model and its tokenizer in FP16."""
    print(f"Loading '{model_name}'. This might take a minute...")
    model_path = BASE_MODEL_ID if model_name == "Base Model (Download/Cache)" else os.path.join(MODELS_DIR, model_name)
        
    tokenizer = AutoTokenizer.from_pretrained(model_path)
    model = AutoModelForCausalLM.from_pretrained(
        model_path,
        torch_dtype=torch.float16,
        device_map="auto"
    )
    model.eval() 
    print(f"Successfully loaded '{model_name}' to {model.device}!\n")
    return model, tokenizer

def generate_answer(model, tokenizer, question, max_tokens=150):
    """Formats the question for Gemma 3 (Conversational QA) and generates a response."""
    messages = [{"role": "user", "content": question}]
    
    prompt = tokenizer.apply_chat_template(
        messages, 
        tokenize=False, 
        add_generation_prompt=True
    )
    
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    input_length = inputs["input_ids"].shape[1]
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id
        )
        
    new_tokens = outputs[0][input_length:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

def generate_continuation(model, tokenizer, prompt, max_tokens=150):
    """Takes a raw text prompt and predicts the verbatim continuation (No Chat Template)."""
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    input_length = inputs["input_ids"].shape[1]
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_tokens,
            # Lower temperature for raw text completion makes verbatim matching easier to spot
            do_sample=True, 
            temperature=0.3, 
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id
        )
        
    new_tokens = outputs[0][input_length:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True)

In [4]:
# List available models
models = get_available_models()
print("Available Models:")
for idx, m_name in enumerate(models):
    print(f"[{idx}] {m_name}")

# Interactive prompt to select a model
choice = input(f"\nSelect a model to load (0-{len(models)-1}): ")
try:
    choice_idx = int(choice)
    selected_model_name = models[choice_idx]
    
    # Clear previous model from memory if it exists to prevent OOM errors
    if 'test_model' in globals():
        del test_model
        torch.cuda.empty_cache()
        
    test_model, test_tokenizer = load_inference_model(selected_model_name)
except (ValueError, IndexError):
    print("Invalid selection. Please run the cell again and enter a valid number.")

Available Models:
[0] Base Model (Download/Cache)
[1] unlearned_gradient_ascent
[2] unlearned_task_vector
[3] fine_tuned_forget



Select a model to load (0-3):  0


Loading 'Base Model (Download/Cache)'. This might take a minute...


config.json:   0%|          | 0.00/899 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/2.00G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/340 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

Successfully loaded 'Base Model (Download/Cache)' to cuda:0!



In [ ]:
print("="*50)
print(f"🧠 Probing: {selected_model_name}")
print("Type 'quit', 'exit', or 'q' to stop.")
print("="*50)

while True:
    user_input = input("\nQuestion: ").strip()
    
    if user_input.lower() in ['quit', 'exit', 'q']:
        print("Ending probe session.")
        break
        
    if not user_input:
        continue
        
    answer = generate_answer(test_model, test_tokenizer, user_input)
    print(f"\nAnswer: {answer}")

In [6]:
print("="*50)
print(f"📖 Raw Text Probing: {selected_model_name}")
print("Paste the beginning of a sentence from the book to see if it can finish it.")
print("Type 'quit', 'exit', or 'q' to stop.")
print("="*50)

while True:
    user_input = input("\nText Snippet: ").strip()
    
    if user_input.lower() in ['quit', 'exit', 'q']:
        print("Ending raw text probe session.")
        break
        
    if not user_input:
        continue
        
    continuation = generate_continuation(test_model, test_tokenizer, user_input)
    
    print(f"\n[Prompt]: {user_input}")
    print(f"[Model Continuation]: {continuation}")

📖 Raw Text Probing: Base Model (Download/Cache)
Paste the beginning of a sentence from the book to see if it can finish it.
Type 'quit', 'exit', or 'q' to stop.



Text Snippet:  how to solve headaches



[Prompt]: how to solve headaches
[Model Continuation]: ?

Okay, let's tackle those headaches! There's no single "fix" for all headaches, as the cause and type of headache can vary. Here's a breakdown of strategies, categorized by severity and potential causes:

**1. Immediate Relief (For Mild to Moderate Headaches)**

* **Rest in a Quiet, Dark Room:**  This is often the most effective first step.  Reduce sensory input.
* **Hydration:** Dehydration is a common trigger. Drink a glass of water.
* **Cold or Warm Compress:**
    * **Cold:**  Apply a cold compress (ice pack wrapped in a towel) to your forehead, temples, or the back of your neck.  Cold const



Text Snippet:  Who is JK Rowling?



[Prompt]: Who is JK Rowling?
[Model Continuation]: 

J.K. Rowling is a British author and businesswoman. She is best known for writing the *Harry Potter* series, which has sold over 500 million copies worldwide.

Here's a breakdown of key aspects of her life and work:

*   **Early Life & Background:** Born in Yate, Gloucestershire, England, in 1965, she grew up in a relatively wealthy family.
*   **Writing Career:** She began writing fantasy novels at a young age, initially focusing on children's stories.
*   **Harry Potter Success:** Her *Harry Potter* series was a global phenomenon, catapulting her to international fame.
*   **Controversy:** Rowling's statements



Text Snippet:  q


Ending raw text probe session.
